In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
(x_train, _), (_, _) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32")
x_train = np.expand_dims(x_train, -1)
x_train = tf.image.resize(x_train, (64, 64)).numpy()
x_train = np.repeat(x_train, 3, axis=-1)
x_train = (x_train - 127.5) / 127.5

In [ ]:
class Generator(tf.keras.Model):
    def __init__(self, latent_dim=100):
        super().__init__()

        self.dense = tf.keras.layers.Dense(16384, use_bias=False)

        self.reshape = tf.keras.layers.Reshape((4, 4, 1024))

        self.bn1 = tf.keras.layers.BatchNormalization()

        self.deconv1 = tf.keras.layers.Conv2DTranspose(
            512, kernel_size=4, strides=2, padding="same", use_bias=False
        )  # 8x8x512
        self.bn2 = tf.keras.layers.BatchNormalization()

        self.deconv2 = tf.keras.layers.Conv2DTranspose(
            256, kernel_size=4, strides=2, padding="same", use_bias=False
        )  # 16x16x256
        self.bn3 = tf.keras.layers.BatchNormalization()

        self.deconv3 = tf.keras.layers.Conv2DTranspose(
            128, kernel_size=4, strides=2, padding="same", use_bias=False
        )  # 32x32x128
        self.bn4 = tf.keras.layers.BatchNormalization()

        self.deconv4 = tf.keras.layers.Conv2DTranspose(
            3, kernel_size=4, strides=2, padding="same", activation="tanh"
        )  # 64x64x3

    def call(self, x, training=False):
        x = self.dense(x)

        x = self.reshape(x)

        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)

        x = self.deconv1(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)

        x = self.deconv2(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)

        x = self.deconv3(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)

        x = self.deconv4(x)

        return x

In [ ]:
class Discriminator(tf.keras.Model):
    def __init__(self):
        super().__init__()

        self.conv1 = tf.keras.layers.Conv2D(
            64, kernel_size=4, strides=2, padding="same"
        )  # 32x32x64
        self.drop1 = tf.keras.layers.Dropout(0.2)

        self.conv2 = tf.keras.layers.Conv2D(
            128, kernel_size=4, strides=2, padding="same"
        )  # 16x16x128
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.drop2 = tf.keras.layers.Dropout(0.2)

        self.conv3 = tf.keras.layers.Conv2D(
            256, kernel_size=4, strides=2, padding="same"
        )  # 8x8x256
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.drop3 = tf.keras.layers.Dropout(0.2)

        self.conv4 = tf.keras.layers.Conv2D(
            512, kernel_size=4, strides=2, padding="same"
        )  # 4x4x512
        self.bn4 = tf.keras.layers.BatchNormalization()

        self.flatten = tf.keras.layers.Flatten()  # 8192
        self.fc = tf.keras.layers.Dense(1)

    def call(self, x, training=False):
        x = self.conv1(x)
        x = tf.nn.leaky_relu(x, alpha=0.2)
        x = self.drop1(x, training=training)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.leaky_relu(x, alpha=0.2)
        x = self.drop2(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.leaky_relu(x, alpha=0.2)
        x = self.drop3(x, training=training)

        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.leaky_relu(x, alpha=0.2)

        x = self.flatten(x)
        x = self.fc(x)

        return x

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy()

def d_loss_fn(real_out, fake_out, real_label=1.0):
    real_loss = bce(tf.ones_like(real_out) * real_label, real_out)
    fake_loss = bce(tf.zeros_like(fake_out), fake_out)
    return real_loss + fake_loss

def g_loss_fn(fake_out):
    return bce(tf.ones_like(fake_out), fake_out)

def d_accuracy(real_out, fake_out):
    real_acc = tf.reduce_mean(tf.cast(real_out > 0.5, tf.float32))
    fake_acc = tf.reduce_mean(tf.cast(fake_out < 0.5, tf.float32))
    return float((real_acc + fake_acc) / 2.0)

In [ ]:
def plot_grid(images, title, n=25):
    images = (images.numpy() + 1) / 2.0  # -> [0, 1], shape (n, 64, 64, 3)
    images = np.clip(images, 0.0, 1.0)
    side = int(np.sqrt(n))
    fig, axes = plt.subplots(side, side, figsize=(6, 6))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i])
        ax.axis('off')
    fig.suptitle(title)
    plt.show()

In [ ]:
def train_dcgan(generator, discriminator, dataset, latent_dim=100, lr=0.02,
                 batch_size=128, epochs=100, label_smoothing=False,
                 snapshot_every=10, g_updates_per_step=1, fixed_latent=None):
    g_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5, beta_2=0.999)
    d_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5, beta_2=0.999)

    real_label = 0.9 if label_smoothing else 1.0
    n_batches = max(1, dataset.shape[0] // batch_size)

    g_losses, d_losses, d_accs = [], [], []
    snapshots = {}

    viz_noise = fixed_latent if fixed_latent is not None else tf.random.normal((25, latent_dim), seed=42)

    for epoch in range(1, epochs + 1):
        idx = np.random.permutation(dataset.shape[0])
        e_g, e_d, e_acc = [], [], []
        for b in range(n_batches):
            real_images = dataset[idx[b * batch_size:(b + 1) * batch_size]]
            real_images = tf.convert_to_tensor(real_images, dtype=tf.float32)
            bs = tf.shape(real_images)[0]

            noise = fixed_latent if fixed_latent is not None else tf.random.normal((bs, latent_dim))
            with tf.GradientTape() as d_tape:
                fake_images = generator(noise, training=True)
                real_out = discriminator(real_images, training=True)
                fake_out = discriminator(fake_images, training=True)
                d_loss = d_loss_fn(real_out, fake_out, real_label)
            d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
            d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))
            acc = d_accuracy(real_out, fake_out)

            g_loss = None
            for _ in range(g_updates_per_step):
                noise = fixed_latent if fixed_latent is not None else tf.random.normal((bs, latent_dim))
                with tf.GradientTape() as g_tape:
                    fake_images = generator(noise, training=True)
                    fake_out = discriminator(fake_images, training=True)
                    g_loss = g_loss_fn(fake_out)
                g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
                g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))

            e_g.append(float(g_loss)); e_d.append(float(d_loss)); e_acc.append(acc)

        g_losses.append(np.mean(e_g)); d_losses.append(np.mean(e_d)); d_accs.append(np.mean(e_acc))
        print(f"Epoch {epoch}/{epochs}  G_loss={g_losses[-1]:.4f}  D_loss={d_losses[-1]:.4f}  D_acc={d_accs[-1]:.4f}")

        if epoch % snapshot_every == 0 or epoch == epochs:
            snapshots[epoch] = generator(viz_noise, training=False)

    return g_losses, d_losses, d_accs, snapshots

In [ ]:
generator = Generator(latent_dim=100)
discriminator = Discriminator()

g_losses_base, d_losses_base, d_accs_base, snaps_base = train_dcgan(
    generator, discriminator, x_train, latent_dim=100, lr=0.02,
    batch_size=128, epochs=10, snapshot_every=2)

In [ ]:
for epoch, imgs in snaps_base.items():
    plot_grid(imgs, f"Baseline DCGAN - Epoch {epoch}")

plt.figure()
plt.plot(g_losses_base, label="Generator Loss")
plt.plot(d_losses_base, label="Discriminator Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Baseline DCGAN - Loss Curves")
plt.legend(); plt.show()

In [ ]:
gen_mod = Generator(latent_dim=200)
disc_mod = Discriminator()

g_losses_mod, d_losses_mod, d_accs_mod, snaps_mod = train_dcgan(
    gen_mod, disc_mod, x_train, latent_dim=200, lr=0.02,
    batch_size=64, epochs=5, label_smoothing=True, snapshot_every=10)

In [ ]:
for epoch, imgs in snaps_mod.items():
    plot_grid(imgs, f"Modified DCGAN - Epoch {epoch}")

plt.figure()
plt.plot(g_losses_mod, label="Generator Loss")
plt.plot(d_losses_mod, label="Discriminator Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Modified DCGAN - Loss Curves")
plt.legend(); plt.show()

In [ ]:
plt.figure()
plt.plot(g_losses_base, label="Baseline G loss")
plt.plot(g_losses_mod, label="Modified G loss")
plt.xlabel("Epoch"); plt.ylabel("Generator Loss")
plt.title("Baseline vs Modified - Generator Loss")
plt.legend(); plt.show()

plt.figure()
plt.plot(d_losses_base, label="Baseline D loss")
plt.plot(d_losses_mod, label="Modified D loss")
plt.xlabel("Epoch"); plt.ylabel("Discriminator Loss")
plt.title("Baseline vs Modified - Discriminator Loss")
plt.legend(); plt.show()

plt.figure()
plt.plot(d_accs_base, label="Baseline D accuracy")
plt.plot(d_accs_mod, label="Modified D accuracy")
plt.xlabel("Epoch"); plt.ylabel("Discriminator Accuracy")
plt.title("Baseline vs Modified - Discriminator Accuracy")
plt.legend(); plt.show()

print("Final baseline: G", g_losses_base[-1], "D", d_losses_base[-1], "acc", d_accs_base[-1])
print("Final modified: G", g_losses_mod[-1], "D", d_losses_mod[-1], "acc", d_accs_mod[-1])

In [ ]:
small_dataset = x_train[np.random.choice(len(x_train), 100, replace=False)]
fixed_z = tf.random.normal((25, 2), seed=42)

gen_collapse = Generator(latent_dim=2)
disc_collapse = Discriminator()

g_losses_collapse, d_losses_collapse, d_accs_collapse, snaps_collapse = train_dcgan(
    gen_collapse, disc_collapse, small_dataset, latent_dim=2, lr=0.005,
    batch_size=25, epochs=5, snapshot_every=10,
    g_updates_per_step=5, fixed_latent=fixed_z)

In [ ]:
for epoch, imgs in snaps_collapse.items():
    plot_grid(imgs, f"Mode Collapse Simulation - Epoch {epoch}")

plt.figure()
plt.plot(g_losses_collapse, label="Generator Loss")
plt.plot(d_losses_collapse, label="Discriminator Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Mode Collapse Simulation - Loss Curves")
plt.legend(); plt.show()

final_samples = snaps_collapse[100].numpy().reshape(25, -1)
diversity_collapse = np.mean(np.std(final_samples, axis=0))
print("Mode collapse setting - pixelwise diversity (std):", diversity_collapse)

final_samples_base = snaps_base[100].numpy().reshape(25, -1)
diversity_base = np.mean(np.std(final_samples_base, axis=0))
print("Baseline setting - pixelwise diversity (std):", diversity_base)

print("Metric        | Baseline | Mode-collapse setting")
print("Final G loss  |", g_losses_base[-1], "|", g_losses_collapse[-1])
print("Final D loss  |", d_losses_base[-1], "|", d_losses_collapse[-1])
print("Diversity     |", diversity_base, "|", diversity_collapse)